# Assignment Beta - Question 1 (SaafHawa) - calculations, student 23K-0003
All numbers in the report come from this notebook. Needs q1_data.py, timing_log.csv and qwen3.8-27b_config.json in the same folder.

In [1]:

import math, csv, statistics, json
from q1_data import M, SOURCES, ACCESS
ROLL='23K-0003'
digits=[int(c) for c in ROLL if c.isdigit()]
R=int(''.join(str(d) for d in digits[-2:])); D=sum(digits)%10
Q=1500+25*R; N=5+(R%4); p=0.04+0.01*D; H=4+(R%4); B=1200+10*R; T=10-(R%3)
print(f"Roll {ROLL}: digits={digits}  R={R}  D=sum({sum(digits)}) mod 10={D}")
print(f"Q={Q}  N={N}  p={p:.2f}  H={H}  B=${B}  T={T}s")


Roll 23K-0003: digits=[2, 3, 0, 0, 0, 3]  R=3  D=sum(8) mod 10=8
Q=1575  N=8  p=0.12  H=7  B=$1230  T=10s


In [2]:

# Section 2.1 token volume
inp=[2000+800*(k-1) for k in range(1,N+1)]
out=[350]*(N-1)+[600]
print('input tokens per call k=1..N:',inp,' sum =',sum(inp))
print('output tokens per call:',out,' sum =',sum(out))
retry_in=p*sum(inp[:N-1]); retry_out=p*sum(out[:N-1])
print(f"retries: each of calls 1..{N-1} fails w.p. p={p:.2f} and is repeated once -> E[extra input]=p*sum(inp[1..N-1])={p:.2f}*{sum(inp[:N-1])}={retry_in:.1f}; E[extra output]=p*{sum(out[:N-1])}={retry_out:.1f}")
E_in=sum(inp)+retry_in; E_out=sum(out)+retry_out; E_calls=N+p*(N-1)
print(f"Expected per query: input={E_in:.1f}, output={E_out:.1f}, LLM calls={E_calls:.2f} (cached-prefix tokens = 1500 x calls = {1500*E_calls:.1f})")
normal_days=30-H
mq=normal_days*Q+H*5*Q
print(f"Monthly queries = {normal_days} x {Q} + {H} x 5 x {Q} = {mq}")
M_in=mq*E_in; M_out=mq*E_out; M_calls=mq*E_calls
print(f"Monthly input tokens = {M_in:,.0f}; output tokens = {M_out:,.0f}; LLM calls = {M_calls:,.0f}")


input tokens per call k=1..N: [2000, 2800, 3600, 4400, 5200, 6000, 6800, 7600]  sum = 38400
output tokens per call: [350, 350, 350, 350, 350, 350, 350, 600]  sum = 3050
retries: each of calls 1..7 fails w.p. p=0.12 and is repeated once -> E[extra input]=p*sum(inp[1..N-1])=0.12*30800=3696.0; E[extra output]=p*2450=294.0
Expected per query: input=42096.0, output=3344.0, LLM calls=8.84 (cached-prefix tokens = 1500 x calls = 13260.0)
Monthly queries = 23 x 1575 + 7 x 5 x 1575 = 91350
Monthly input tokens = 3,845,469,600; output tokens = 305,474,400; LLM calls = 807,534


In [3]:

# Section 2.2 API cost, with / without caching (prefix 1500 tokens; reasoning overhead base case = 0, see note)
PREFIX=1500
cached_per_query=PREFIX*E_calls
def cost(m, extra_out_per_call=0.0):
    out_tok=M_out+M_calls*extra_out_per_call
    nocache=(M_in*m['pin']+out_tok*m['pout'])/1e6
    ok,pr,pw,mn,ttl,_=m['cache']
    if (not ok) or (mn is not None and PREFIX<mn):
        return nocache,nocache,0.0,0.0,False
    reads=mq*cached_per_query
    windows=(30*24*60/ttl) if ttl else 30*24*60/5   # conservative: one write per TTL window (real cost lower: reads refresh the TTL)
    writes=windows*PREFIX
    reads-=writes
    uncached=M_in-reads-writes
    c=(uncached*m['pin']+reads*pr+writes*pw+out_tok*m['pout'])/1e6
    return nocache,c,reads,writes,True
rows=[]
for m in M:
    nc,c,rd,wr,used=cost(m)
    nc2,c2,_,_,_=cost(m,300.0)   # sensitivity: +300 reasoning tokens per call
    rows.append(dict(m=m,nocache=nc,cache=c,used=used,reads=rd,writes=wr,sens=c2))
    print(f"{m['name']:32s} no-cache ${nc:10,.2f} | cache ${c:10,.2f} (cache applied: {used}) | within B? no-cache={nc<=B} cache={c<=B} | +300 reasoning tok/call: ${c2:,.2f}")
print('Within B without caching:',[r['m']['name'] for r in rows if r['nocache']<=B])
print('Within B with caching   :',[r['m']['name'] for r in rows if r['cache']<=B])


Claude Haiku 5.5                 no-cache $    537.28 | cache $    429.76 (cache applied: True) | within B? no-cache=True cache=True | +300 reasoning tok/call: $550.89
Claude Sonnet 5.5                no-cache $ 10,745.68 | cache $  8,475.32 (cache applied: True) | within B? no-cache=False cache=False | +300 reasoning tok/call: $10,897.92
GPT-6 Luna                       no-cache $    537.28 | cache $    428.52 (cache applied: True) | within B? no-cache=True cache=True | +300 reasoning tok/call: $549.65
GPT-6.1 Terra                    no-cache $  5,372.84 | cache $  4,224.70 (cache applied: True) | within B? no-cache=False cache=False | +300 reasoning tok/call: $5,436.00
Gemini 3.8 Flash                 no-cache $  4,432.59 | cache $  4,432.59 (cache applied: False) | within B? no-cache=False cache=False | +300 reasoning tok/call: $5,431.92
Grok 4.7                         no-cache $  9,523.79 | cache $  7,700.35 (cache applied: True) | within B? no-cache=False cache=False | +300 reas

In [4]:

# Section 2.3 peak load
day_q=5*Q; win_q=0.35*day_q; qpm=win_q/120
print(f"Heatwave day queries = 5 x {Q} = {day_q}; 35% in 12:00-14:00 = {win_q:.2f} queries in 120 min -> {qpm:.2f} queries/min")
rpm=qpm*E_calls; itpm=qpm*E_in; otpm=qpm*E_out
print(f"Peak LLM requests/min = {qpm:.2f} x {E_calls:.2f} = {rpm:.1f}; input tokens/min = {itpm:,.0f}; output tokens/min = {otpm:,.0f}; total TPM = {itpm+otpm:,.0f}")
print(f"Uncached ITPM if prefix cached (Anthropic counts only uncached) = {qpm*(E_in-PREFIX*E_calls):,.0f}")
print('vs Anthropic Start: 4000 RPM / 2,000,000 ITPM / 400,000 OTPM ->', 'OK' if rpm<4000 and itpm<2e6 and otpm<4e5 else 'EXCEEDED')
print('vs OpenAI Luna Build: 5000 RPM / 2M TPM ->','OK' if rpm<5000 and itpm+otpm<2e6 else 'EXCEEDED')
print('vs OpenAI Terra Build (if 1M TPM applies): 5000 RPM / 1M TPM ->','OK' if rpm<5000 and itpm+otpm<1e6 else 'EXCEEDED (total TPM %.0f > 1,000,000)'%(itpm+otpm))


Heatwave day queries = 5 x 1575 = 7875; 35% in 12:00-14:00 = 2756.25 queries in 120 min -> 22.97 queries/min
Peak LLM requests/min = 22.97 x 8.84 = 203.0; input tokens/min = 966,892; output tokens/min = 76,808; total TPM = 1,043,700
Uncached ITPM if prefix cached (Anthropic counts only uncached) = 662,327
vs Anthropic Start: 4000 RPM / 2,000,000 ITPM / 400,000 OTPM -> OK
vs OpenAI Luna Build: 5000 RPM / 2M TPM -> OK
vs OpenAI Terra Build (if 1M TPM applies): 5000 RPM / 1M TPM -> EXCEEDED (total TPM 1043700 > 1,000,000)


In [5]:

# Section 2.4 self-hosting memory: Qwen3.8-27B (hybrid) from config.json
cfg=json.load(open('qwen3.8-27b_config.json'))['text_config']
layers=cfg['num_hidden_layers']; kvh=cfg['num_key_value_heads']; hd=cfg['head_dim']
full=cfg['layer_types'].count('full_attention'); lin=layers-full
print(f"config.json: num_hidden_layers={layers}, num_key_value_heads={kvh}, head_dim={hd}, full_attention layers={full}, linear_attention layers={lin}")
PARAMS=27e9
GB=1e9
w16=PARAMS*2/GB; w8=PARAMS*1/GB; w4=PARAMS*0.5/GB
print(f"Weights: FP16={w16:.1f} GB, 8-bit={w8:.1f} GB, 4-bit={w4:.1f} GB (27e9 params; 1 GB=1e9 B; quantisation scales ignored)")
seq=inp[-1]+out[-1]
print(f"Longest context in agent loop = last-call input {inp[-1]} + output {out[-1]} = {seq} tokens")
# only the full-attention layers keep a growing KV cache; formula adapted: layers -> full_attention layers
kv=2*full*kvh*hd*seq*2
print(f"KV bytes/seq = 2 x {full} x {kvh} x {hd} x {seq} x 2 = {kv:,} B = {kv/GB:.3f} GB  ({kv/seq:.0f} B/token)")
# fixed recurrent state of Gated-DeltaNet layers (does not grow with length)
vh=cfg['linear_num_value_heads']; kd=cfg['linear_key_head_dim']; vd=cfg['linear_value_head_dim']; ck=cfg['linear_conv_kernel_dim']
conv_ch=2*cfg['linear_num_key_heads']*kd+vh*vd
state=lin*(vh*kd*vd*4 + conv_ch*(ck-1)*2)
print(f"Linear-attention state/seq = {lin} x ({vh}x{kd}x{vd}x4 B fp32 + {conv_ch}x{ck-1}x2 B conv) = {state:,} B = {state/GB:.3f} GB (independent of length; mamba_ssm_dtype=float32)")
per_seq=kv+state
GPU_GB=80; usable=GPU_GB*0.9
free=usable-w4
conc=math.floor(free*GB/per_seq)
print(f"GPU: NVIDIA H100 PCIe 80 GB. Usable = 0.9 x 80 = {usable:.1f} GB; minus 4-bit weights {w4:.1f} = {free:.1f} GB for caches")
print(f"Per-sequence cache = {per_seq/GB:.3f} GB -> max concurrent sequences = floor({free:.1f}/{per_seq/GB:.3f}) = {conc}   (KV-only: {math.floor(free*GB/kv)})")
LAT_SLA=T
peak_conc=qpm/60*LAT_SLA
print(f"Peak concurrency (Little's law) = {qpm/60:.4f} q/s x {LAT_SLA}s = {peak_conc:.2f} -> {math.ceil(peak_conc)} sequences")
gpus=max(1,math.ceil(peak_conc/conc)); print('GPUs required =',gpus)


config.json: num_hidden_layers=64, num_key_value_heads=4, head_dim=256, full_attention layers=16, linear_attention layers=48
Weights: FP16=54.0 GB, 8-bit=27.0 GB, 4-bit=13.5 GB (27e9 params; 1 GB=1e9 B; quantisation scales ignored)
Longest context in agent loop = last-call input 7600 + output 600 = 8200 tokens
KV bytes/seq = 2 x 16 x 4 x 256 x 8200 x 2 = 537,395,200 B = 0.537 GB  (65536 B/token)
Linear-attention state/seq = 48 x (48x128x128x4 B fp32 + 10240x3x2 B conv) = 153,944,064 B = 0.154 GB (independent of length; mamba_ssm_dtype=float32)
GPU: NVIDIA H100 PCIe 80 GB. Usable = 0.9 x 80 = 72.0 GB; minus 4-bit weights 13.5 = 58.5 GB for caches
Per-sequence cache = 0.691 GB -> max concurrent sequences = floor(58.5/0.691) = 84   (KV-only: 108)
Peak concurrency (Little's law) = 0.3828 q/s x 10s = 3.83 -> 4 sequences
GPUs required = 1


In [6]:

# Section 2.6 tool timing (pooled median of all measurements, per assignment)
rows_t=list(csv.DictReader(open('timing_log.csv')))
allv=[float(r['latency_s']) for r in rows_t if r['ok']=='1']
days=sorted(set(r['date'] for r in rows_t))
net=[float(r['latency_s']) for r in rows_t if r['ok']=='1' and r['tool'].endswith('open_meteo')]
tool_delay=statistics.median(allv)
print(f"measurements={len(allv)} over days {days}; per-tool counts:",{t:sum(1 for r in rows_t if r['tool']==t) for t in sorted(set(r['tool'] for r in rows_t))})
print(f"pooled median (all tools, as specified) = {tool_delay:.4f} s ; network-only median = {statistics.median(net):.4f} s")
def lat_query(m,calls_out,delay,stages,streamed=False,tps_override=None,ttft_override=None):
    ttft=ttft_override or m['ttft']; tps=tps_override or m['tps']
    per=[ttft+o/tps for o in calls_out]
    if streamed: per[-1]=ttft            # user sees first answer token after final TTFT
    base=sum(per); retry=p*sum(per[:-1]); tools=stages*delay
    return base,retry,tools,base+retry+tools
cands=[m for m in M if m['id'] in ('haiku','sonnet','luna','terra','gemini','grok','dsv4','qwen','gemma','mistral','glm')]
print('Baseline sequential N=%d (TTFT/speed from AA at the chosen setting):'%N)
for m in cands:
    b,r,t,tot=lat_query(m,out,tool_delay,N-1); print(f"  {m['name']:28s} calls={b:6.1f}s retries={r:5.1f}s tools={t:4.2f}s total={tot:6.1f}s {'MEETS' if tot<=T else 'exceeds'} T={T}s")


measurements=100 over days ['2026-11-16']; per-tool counts: {'air_quality_open_meteo': 25, 'heat_index_local': 25, 'kb_lookup_local': 25, 'weather_open_meteo': 25}
pooled median (all tools, as specified) = 0.2491 s ; network-only median = 0.5747 s
Baseline sequential N=8 (TTFT/speed from AA at the chosen setting):
  Claude Haiku 5.5             calls=  56.0s retries=  5.8s tools=1.74s total=  63.5s exceeds T=10s
  Claude Sonnet 5.5            calls=  43.1s retries=  4.3s tools=1.74s total=  49.1s exceeds T=10s
  GPT-6 Luna                   calls=  46.2s retries=  4.6s tools=1.74s total=  52.5s exceeds T=10s
  GPT-6.1 Terra                calls=  64.6s retries=  6.4s tools=1.74s total=  72.7s exceeds T=10s
  Gemini 3.8 Flash             calls=  63.2s retries=  6.4s tools=1.74s total=  71.3s exceeds T=10s
  Grok 4.7                     calls= 131.6s retries= 13.4s tools=1.74s total= 146.7s exceeds T=10s
  DeepSeek V4.1 Flash 1017     calls=  30.6s retries=  3.0s tools=1.74s total=  35.4

In [7]:

# Candidates: strongest tool-use evidence among models that fit budget B (and have published speed). Chosen after the cost table:
#   (1) Claude Haiku 5.5 (closed, in budget, Terminal-Bench 4.0 = 36.4%)  (2) DeepSeek V4.1 Flash 1017 (open, MIT, in budget, TB 4.0 = 31.8%, fastest TTFT)
# (Sonnet 5.5 / Terra are stronger but exceed B; GLM-5.3-Flash has 90 s TTFT.)
H5=[m for m in M if m['id']=='haiku'][0]; DS=[m for m in M if m['id']=='dsv4'][0]
N2=4; out2=[350]*(N2-1)+[600]
print('ARCH CHANGE: run the independent tools (air-quality, weather, KB) in PARALLEL inside one LLM turn.')
print(f'LLM calls fall {N}->{N2}: (1) identify area/time/person + plan, (2) parallel AQ+weather+KB, (3) heat index, (4) answer; tool stages = {N2-1} (not {N-1}).')
res={}
for m in (H5,DS):
    b0,r0,t0,tot0=lat_query(m,out,tool_delay,N-1)
    b1,r1,t1,tot1=lat_query(m,out2,tool_delay,N2-1)
    b2,r2,t2,tot2=lat_query(m,out2,tool_delay,N2-1,streamed=True)
    res[m['id']]=(tot0,tot1,tot2)
    print(f"{m['name']}: baseline {tot0:.1f}s -> parallel-tools {tot1:.1f}s (calls {b1:.1f}+retries {r1:.1f}+tools {t1:.2f}) -> +streaming final {tot2:.1f}s ; T={T}s")
# Extra: route steps 1-3 to a fast small model (GPT-6 Luna NR: TTFT 1.54s, 90 tok/s) and keep the strong model for the final answer
LU=[m for m in M if m['id']=='luna'][0]
for m in (H5,DS):
    per_small=[LU['ttft']+o/LU['tps'] for o in out2[:-1]]; fin=m['ttft']
    tot=sum(per_small)+p*sum(per_small)+(N2-1)*tool_delay+fin
    print(f"{m['name']}: small-model routing (Luna for calls 1-3) + parallel tools + streamed final = {tot:.1f}s")


ARCH CHANGE: run the independent tools (air-quality, weather, KB) in PARALLEL inside one LLM turn.
LLM calls fall 8->4: (1) identify area/time/person + plan, (2) parallel AQ+weather+KB, (3) heat index, (4) answer; tool stages = 3 (not 7).
Claude Haiku 5.5: baseline 63.5s -> parallel-tools 31.8s (calls 28.6+retries 2.5+tools 0.75) -> +streaming final 28.9s ; T=10s
DeepSeek V4.1 Flash 1017: baseline 35.4s -> parallel-tools 18.2s (calls 16.1+retries 1.3+tools 0.75) -> +streaming final 14.2s ; T=10s
Claude Haiku 5.5: small-model routing (Luna for calls 1-3) + parallel tools + streamed final = 24.2s
DeepSeek V4.1 Flash 1017: small-model routing (Luna for calls 1-3) + parallel tools + streamed final = 20.3s


In [8]:

# Section 2.5 break-even
PRICE=3.79; HRS=720; gpu_month=PRICE*HRS
print(f"H100 PCIe 80GB at ${PRICE}/hr x {HRS} h = ${gpu_month:,.2f} per GPU-month (Lambda, accessed {ACCESS})")
def api_cost_for_daily(m,d,sens=0):
    # scale the monthly volume linearly with normal-day volume d (heatwave day = 5d)
    k=d/Q; nc,c,_,_,_=cost(m,sens); return c*k
def gpus_for_daily(d):
    qpm_d=0.35*5*d/120
    return max(1,math.ceil(qpm_d/60*LAT_SLA/conc))
def selfhost_cost(d): return gpus_for_daily(d)*gpu_month
cand_rows={m['id']:[r for r in rows if r['m']['id']==m['id']][0] for m in (H5,DS)}
for m in (H5,DS):
    print(m['name'],'monthly API cost (cached) at Q=%d: $%.2f'%(Q,cand_rows[m['id']]['cache']))
# Which candidate meets T? use latency after the architectural change (parallel tools + streaming)
meets=[m for m in (H5,DS) if res[m['id']][2]<=T]
if len(meets)==0:
    ref=min((H5,DS),key=lambda m:res[m['id']][2]); print('Neither meets T after change -> use lower latency:',ref['name'])
elif len(meets)==1: ref=meets[0]; print('Only one meets T:',ref['name'])
else:
    ref=min(meets,key=lambda m:cand_rows[m['id']]['cache']); print('Both meet T -> cheaper:',ref['name'])
def qstar(m):
    for d in range(100,2_000_000,50):
        if selfhost_cost(d)<api_cost_for_daily(m,d): return d
    return None
Qstar=qstar(ref); print('Reference API option:',ref['name'],' Q* (self-host cheaper) =',Qstar,'queries/day  (= %.1f x Q)'%(Qstar/Q if Qstar else float('nan')))
for m in (H5,DS):
    print('  vs',m['name'],'Q* =',qstar(m))
step_edges=[(g,g*conc*60/LAT_SLA/(0.35*5/120)) for g in range(1,4)]
print('GPU step capacity: 1 GPU covers up to %.0f queries/day, 2 GPUs up to %.0f, 3 GPUs up to %.0f (assuming %ds per query)'%(step_edges[0][1],step_edges[1][1],step_edges[2][1],LAT_SLA))


H100 PCIe 80GB at $3.79/hr x 720 h = $2,728.80 per GPU-month (Lambda, accessed 2026-11-16)
Claude Haiku 5.5 monthly API cost (cached) at Q=1575: $429.76
DeepSeek V4.1 Flash 1017 monthly API cost (cached) at Q=1575: $171.29
Neither meets T after change -> use lower latency: DeepSeek V4.1 Flash 1017
Reference API option: DeepSeek V4.1 Flash 1017  Q* (self-host cheaper) = 25100 queries/day  (= 15.9 x Q)
  vs Claude Haiku 5.5 Q* = 10050
  vs DeepSeek V4.1 Flash 1017 Q* = 25100
GPU step capacity: 1 GPU covers up to 34560 queries/day, 2 GPUs up to 69120, 3 GPUs up to 103680 (assuming 10s per query)
